# Map the Menche interactome to gene symbols (via Entrez IDs)

The gene IDs in `DataS1_interactome.tsv` are **Entrez Gene IDs** (NCBI), e.g. `7157` = TP53.
They are *not* HGNC IDs (`HGNC:7157` = MMP11), so we map them with the `entrez_id` column of the HGNC table.

In [1]:
from pathlib import Path

import pandas as pd

OUT_DIR = Path("outputs/interactome")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the TSV interactome dataset
interactome_df = pd.read_csv(
    "../our_paper/sep_25/DataS1_interactome.tsv",
    sep="\t",
    comment="#",   # ignore all lines starting with '#'
    header=None,
    names=["gene_ID_1", "gene_ID_2", "data_source"],
)

# Load the HGNC complete set dataset
hgnc_df = pd.read_csv("./inputs/hgnc_complete_set.txt", sep="\t", encoding="utf-8", low_memory=False)

# Optional: preview the data
print("Interactome dataset:")
interactome_df.head()

Interactome dataset:


,gene_ID_1,gene_ID_2,data_source
0,1394,2778,literature
1,100290337,4214,literature
2,122704,54460,complexes
3,4790,79155,binary;literature;signaling
4,2597,70,signaling


In [2]:
list(interactome_df.columns)

['gene_ID_1', 'gene_ID_2', 'data_source']

In [3]:
cols = [f"{i}:{c}" for i, c in enumerate(hgnc_df.columns)]
width = max(map(len, cols)) + 2
per_row = max(1, 150 // width)   # 150 = total line width
for r in range(0, len(cols), per_row):
    print("".join(c.ljust(width) for c in cols[r:r + per_row]))

0:hgnc_id                    1:symbol                     2:name                       3:locus_group                4:locus_type                 
5:status                     6:location                   7:alias_symbol               8:alias_name                 9:prev_symbol                
10:prev_name                 11:gene_group                12:gene_group_id             13:date_approved_reserved    14:date_symbol_changed       
15:date_name_changed         16:date_modified             17:entrez_id                 18:ensembl_gene_id           19:vega_id                   
20:ucsc_id                   21:ena                       22:refseq_accession          23:ccds_id                   24:uniprot_ids               
25:pubmed_id                 26:mgd_id                    27:rgd_id                    28:lsdb                      29:cosmic                    
30:omim_id                   31:mirbase                   32:homeodb                   33:snornabase                34:biopa

In [4]:
print("\nHGNC dataset:")
hgnc_df[["hgnc_id", "symbol", "entrez_id"]].head()


HGNC dataset:


,hgnc_id,symbol,entrez_id
0,HGNC:5,A1BG,1.0
1,HGNC:37133,A1BG-AS1,503538.0
2,HGNC:24086,A1CF,29974.0
3,HGNC:7,A2M,2.0
4,HGNC:27057,A2M-AS1,144571.0


## Build the Entrez ID → symbol map
`entrez_id` is read as a float (because some genes have no Entrez ID), so we drop the missing ones and convert to int.

In [5]:
entrez_symbol_map = (
    hgnc_df[["entrez_id", "symbol"]]
    .dropna(subset=["entrez_id", "symbol"])
    .astype({"entrez_id": int})
    .set_index("entrez_id")["symbol"]
    .to_dict()
)
print(f"Entrez IDs in HGNC map: {len(entrez_symbol_map)}")

Entrez IDs in HGNC map: 44432


In [6]:
interactome_df["symbol_1"] = interactome_df["gene_ID_1"].map(entrez_symbol_map)
interactome_df["symbol_2"] = interactome_df["gene_ID_2"].map(entrez_symbol_map)

all_ids = pd.unique(interactome_df[["gene_ID_1", "gene_ID_2"]].values.ravel())
n_mapped = sum(i in entrez_symbol_map for i in all_ids)
print(f"Genes in interactome: {len(all_ids)}")
print(f"Genes with a symbol:  {n_mapped} ({n_mapped / len(all_ids):.1%})")

Genes in interactome: 13460
Genes with a symbol:  13334 (99.1%)


## Unmapped interactions out of the total (raw file)
How many interactions lose at least one gene because its Entrez ID has no symbol in HGNC?

In [7]:
def unmapped_summary(df, label):
    """Print how many interactions and genes in df have no symbol."""
    miss_1 = df["symbol_1"].isna()
    miss_2 = df["symbol_2"].isna()
    n_total = len(df)
    n_unmapped = (miss_1 | miss_2).sum()

    ids = pd.unique(df[["gene_ID_1", "gene_ID_2"]].values.ravel())
    n_unmapped_genes = sum(i not in entrez_symbol_map for i in ids)

    print(f"--- {label} ---")
    print(f"Interactions:          {n_total}")
    print(f"Unmapped interactions: {n_unmapped} ({n_unmapped / n_total:.2%})")
    print(f"  one gene unmapped:   {(miss_1 ^ miss_2).sum()}")
    print(f"  both genes unmapped: {(miss_1 & miss_2).sum()}")
    print(f"Genes:                 {len(ids)}")
    print(f"Unmapped genes:        {n_unmapped_genes} ({n_unmapped_genes / len(ids):.2%})")


# An interaction is "unmapped" when at least one of its two genes has no symbol
unmapped_summary(interactome_df, "Raw file")

--- Raw file ---
Interactions:          141296
Unmapped interactions: 575 (0.41%)
  one gene unmapped:   562
  both genes unmapped: 13
Genes:                 13460
Unmapped genes:        126 (0.94%)


## Extract the unmapped genes
Save the Entrez IDs that have no symbol, with how many interactions each one has.
This runs on the raw file (before self-interactions are removed), so it lists all 126 unmapped genes.

In [8]:
# Entrez IDs in the interactome that have no symbol in HGNC
unmapped_ids = sorted(int(i) for i in all_ids if i not in entrez_symbol_map)

# One row per (gene, interaction), so we can count the interactions of each gene
is_self = interactome_df["gene_ID_1"] == interactome_df["gene_ID_2"]
side_1 = interactome_df[["gene_ID_1", "data_source"]].set_axis(["entrez_id", "data_source"], axis=1)
side_2 = interactome_df[["gene_ID_2", "data_source"]].set_axis(["entrez_id", "data_source"], axis=1)
per_gene = pd.concat([side_1.assign(is_self=is_self), side_2[~is_self].assign(is_self=False)])
per_gene = per_gene[per_gene["entrez_id"].isin(unmapped_ids)]

unmapped_genes_df = per_gene.groupby("entrez_id").agg(
    interactions_raw=("is_self", "size"),
    interactions_excl_self=("is_self", lambda s: (~s).sum()),
    self_interaction=("is_self", "any"),
    # all the evidence types that support the gene's interactions
    data_sources=("data_source", lambda s: ";".join(sorted(set(";".join(s).split(";"))))),
).reset_index()

# False = the gene only interacts with itself, so it disappears when self-interactions are removed
unmapped_genes_df.insert(
    4, "present_after_self_interaction_removal", unmapped_genes_df["interactions_excl_self"] > 0
)

# One Entrez ID per line
ids_path = OUT_DIR / "unmapped_gene_ids.txt"
ids_path.write_text("\n".join(map(str, unmapped_ids)) + "\n")

csv_path = OUT_DIR / "unmapped_genes.csv"
unmapped_genes_df.to_csv(csv_path, index=False)

print(f"Saved {len(unmapped_ids)} unmapped Entrez IDs to: {ids_path}")
print(f"Saved their details to: {csv_path}")
unmapped_genes_df.sort_values("interactions_excl_self", ascending=False).head(10)

Saved 126 unmapped Entrez IDs to: outputs/interactome/unmapped_gene_ids.txt
Saved their details to: outputs/interactome/unmapped_genes.csv


,entrez_id,interactions_raw,interactions_excl_self,self_interaction,present_after_self_interaction_removal,data_sources
103,732272,69,68,True,True,binary
113,100290337,34,34,False,True,literature
105,100124696,27,26,True,True,literature
10,4950,25,24,True,True,binary;kinase;literature;signaling
54,127557,17,17,False,True,signaling
67,285216,16,16,False,True,metabolic
84,440321,16,16,False,True,binary
46,85287,14,14,False,True,binary
85,440552,13,13,False,True,binary;signaling
2,1231,12,11,True,True,literature;signaling


In [9]:
# Optional (needs internet): ask NCBI Gene why each ID has no symbol.
#   replaced     = the gene now has a newer Entrez ID (which can be mapped to a symbol)
#   discontinued = the record was withdrawn
#   live         = the record exists at NCBI but HGNC has no entry for it
import json
from urllib.request import urlopen

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=gene&retmode=json&id="
    + ",".join(map(str, unmapped_ids))
)
with urlopen(url, timeout=60) as response:
    ncbi = json.load(response)["result"]

STATUS = {"": "live", "0": "live", "1": "replaced", "2": "discontinued"}
network_ids = set(all_ids)

ncbi_rows = []
for gene_id in unmapped_genes_df["entrez_id"]:
    record = ncbi[str(gene_id)]
    new_id = int(record["currentid"]) if str(record.get("currentid", "")).strip() not in ("", "0") else None
    ncbi_rows.append({
        "ncbi_symbol": record.get("name", ""),
        "ncbi_description": record.get("description", ""),
        "ncbi_status": STATUS[str(record.get("status", ""))],
        "replaced_by_entrez_id": new_id,
        "replacement_hgnc_symbol": entrez_symbol_map.get(new_id),
        "organism": record.get("organism", {}).get("scientificname", ""),
        # True = the new ID is already a gene in the interactome
        "replacement_already_in_interactome": (new_id in network_ids) if new_id else None,
    })

unmapped_genes_df = pd.concat([unmapped_genes_df.iloc[:, :6], pd.DataFrame(ncbi_rows)], axis=1)
unmapped_genes_df["replaced_by_entrez_id"] = unmapped_genes_df["replaced_by_entrez_id"].astype("Int64")
unmapped_genes_df.to_csv(csv_path, index=False)

print(unmapped_genes_df["ncbi_status"].value_counts().to_string())
print(f"\nSaved NCBI details to: {csv_path}")

ncbi_status
replaced        74
discontinued    31
live            21

Saved NCBI details to: outputs/interactome/unmapped_genes.csv


## Sanity check
Well-known interacting protein pairs should be found in the network after mapping.

In [10]:
known_pairs = [("TP53", "MDM2"), ("BRCA1", "BARD1"), ("BRCA2", "PALB2"), ("ESR1", "NCOA3"), ("CDK4", "CCND1")]

edge_set = set(map(frozenset, zip(interactome_df["symbol_1"], interactome_df["symbol_2"])))
for a, b in known_pairs:
    print(f"{a} - {b}: {'found' if frozenset((a, b)) in edge_set else 'NOT found'}")

TP53 - MDM2: found
BRCA1 - BARD1: found
BRCA2 - PALB2: found
ESR1 - NCOA3: found
CDK4 - CCND1: found


In [11]:
len_before = len(interactome_df)
print(f"Rows before removing self-interactions: {len_before}")

# Remove rows where gene_ID_1 and gene_ID_2 are identical
interactome_df = interactome_df[
    interactome_df["gene_ID_1"] != interactome_df["gene_ID_2"]
].copy()
len_after = len(interactome_df)
change_percentage = (len_after - len_before ) / len_before

print(f"Rows remaining after removing self-interactions: {len_after}")
print(f"Percentage Change: {change_percentage:.2%}")


Rows before removing self-interactions: 141296
Rows remaining after removing self-interactions: 138427
Percentage Change: -2.03%


In [12]:
interactome_df = interactome_df.copy()

print(f"Rows before removing duplicate gene pairs: {len(interactome_df)}")

# Treat (gene_ID_1, gene_ID_2) and (gene_ID_2, gene_ID_1) as the same interaction
interactome_df["pair_key"] = interactome_df[["gene_ID_1", "gene_ID_2"]].apply(
    lambda row: tuple(sorted((int(row["gene_ID_1"]), int(row["gene_ID_2"])))), axis=1
)

interactome_df = interactome_df.drop_duplicates(subset="pair_key", keep="first").drop(columns="pair_key")

print(f"Rows after removing duplicate gene pairs: {len(interactome_df)}")

Rows before removing duplicate gene pairs: 138427
Rows after removing duplicate gene pairs: 138427


In [13]:
interactome_df = interactome_df[['symbol_1', 'symbol_2', 'gene_ID_1', 'gene_ID_2', 'data_source']]
interactome_df

,symbol_1,symbol_2,gene_ID_1,gene_ID_2,data_source
0,CRHR1,GNAS,1394,2778,literature
1,NaN,MAP3K1,100290337,4214,literature
2,MRPL52,MRPS21,122704,54460,complexes
3,NFKB1,TNIP2,4790,79155,binary;literature;signaling
4,GAPDH,ACTC1,2597,70,signaling
...,...,...,...,...,...
141291,MRPL9,MRPL45,65005,84311,complexes
141292,NEK1,TSC2,4750,7249,binary;literature;signaling
141293,MYO5A,BMF,4644,90427,literature
141294,HIPK1,FHL3,204851,2275,binary


In [14]:
interactome_df.to_csv(OUT_DIR / "interactome_all.csv", index=False)

## Interactions where both genes have a symbol

In [15]:
both_symbols_df = interactome_df.dropna(subset=["symbol_1", "symbol_2"]).copy()
print(f"Interactions with both symbols: {len(both_symbols_df)} of {len(interactome_df)}")

both_symbols_df.to_csv(OUT_DIR / "interactome_both_symbols.csv", index=False)

Interactions with both symbols: 137863 of 138427


In [16]:
# Find rows where either symbol_1 or symbol_2 is missing,
# then export them to Excel and highlight the corresponding gene_ID column(s)
from openpyxl.styles import PatternFill

missing_rows = interactome_df[
    interactome_df[["symbol_1", "symbol_2"]].isna().any(axis=1)
].copy()

# Keep only useful columns for export
missing_rows = missing_rows[
    ["symbol_1", "symbol_2", "gene_ID_1", "gene_ID_2", "data_source"]
].copy()

excel_path = OUT_DIR / "missing_symbols.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    missing_rows.to_excel(writer, index=False, sheet_name="missing_symbols")

    ws = writer.sheets["missing_symbols"]

    # Highlight gene_ID_1 if symbol_1 is missing
    # Highlight gene_ID_2 if symbol_2 is missing
    yellow_fill = PatternFill(fill_type="solid", fgColor="FFD966")

    for row_idx in range(2, len(missing_rows) + 2):
        if pd.isna(missing_rows.iloc[row_idx - 2]["symbol_1"]):
            ws.cell(row=row_idx, column=3).fill = yellow_fill  # gene_ID_1 column
        if pd.isna(missing_rows.iloc[row_idx - 2]["symbol_2"]):
            ws.cell(row=row_idx, column=4).fill = yellow_fill  # gene_ID_2 column

print(f"Saved {len(missing_rows)} rows with missing symbols to: {excel_path}")

Saved 564 rows with missing symbols to: outputs/interactome/missing_symbols.xlsx


## Unmapped interactions out of the total (final network)
Same count as above, but on the network the pipeline actually uses (self-interactions removed).
Dropping the unmapped interactions can also leave a mapped gene with no interactions at all, so we count the genes that remain.

In [17]:
unmapped_summary(interactome_df, "After removing self-interactions")

genes_kept = pd.unique(both_symbols_df[["gene_ID_1", "gene_ID_2"]].values.ravel())
print(f"\nMapped network: {len(both_symbols_df)} interactions, {len(genes_kept)} genes")

--- After removing self-interactions ---
Interactions:          138427
Unmapped interactions: 564 (0.41%)
  one gene unmapped:   562
  both genes unmapped: 2
Genes:                 13397
Unmapped genes:        124 (0.93%)

Mapped network: 137863 interactions, 13270 genes
